# Punting

Everything this warehouse knows about punters, as live queries. Re-run the notebook and the numbers
update — nothing below is transcribed by hand.

Punter is the odd position out: the ESPN league starts one (lineup slot 18) and no external source
in this warehouse prices it. FantasyPros, CBS, FFToday, FFC and Sleeper don't rank punters at all.
So everything here is built from nflverse weekly stats and the punt play-by-play archive.

**Contents**
1. [What the league actually scores](#scoring)
2. [Is punting predictable at all?](#predictable)
3. [Are bad teams' punters better?](#badteams)
4. [Does punting more lower your average?](#tension)
5. [Does matchup matter?](#matchup)
6. [The model, and how well it does](#model)
7. [This year's projections](#projections)

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.query import q, tables, columns, peek

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

tables("gold").query("table.str.startswith('punt')")

,table,layer,rows
17,punt_environment,gold,344
18,punter_backtest,gold,28
19,punter_projections,gold,35
20,punter_seasons,gold,431


<a id="scoring"></a>
## 1. What the league actually scores

Ten punting categories, read out of `league_settings` rather than hardcoded. Two things about this
scoring drive everything else:

- **Inside-the-10 stacks on inside-the-20.** A punt downed at the 6 scores both, so it's worth 5,
  not 3.
- **The average bonus is per *game*, not per season.** A punter earns it once for each game he
  finishes in that gross-average band, which is why scoring has to happen week by week.

In [2]:
q("""
    SELECT
        punt_pts            AS "per punt",
        punt_in10_pts       AS "inside 10",
        punt_in20_pts       AS "inside 20",
        punt_returned_pts   AS "returned",
        punt_touchback_pts  AS "touchback",
        punt_fair_catch_pts AS "fair catch",
        punt_blocked_pts    AS "blocked",
        punt_avg44_pts      AS "game avg 44+",
        punt_avg42_pts      AS "game avg 42-44",
        punt_avg40_pts      AS "game avg 40-42",
        p_slots             AS "starting slots"
    FROM league_settings WHERE league_key = 'espn'
""").T.rename(columns={0: "points"})

,points
per punt,1.0
inside 10,3.0
inside 20,2.0
returned,-1.0
touchback,-1.0
fair catch,0.5
blocked,-1.0
game avg 44+,3.0
game avg 42-44,2.0
game avg 40-42,1.0


For scale, ESPN's own 2026 projections put punters *ahead of kickers* in this league — and the
gap from the best starter to the tenth is small either way. This is a last-round position; the
value in modelling it is not drafting badly, not finding an edge.

In [3]:
q("""
    SELECT position,
           COUNT(*)                                   AS players,
           ROUND(MAX(projected_points), 1)            AS best,
           ROUND(QUANTILE_CONT(projected_points, 0.5), 1) AS median
    FROM espn_projections
    WHERE season = 2026 AND projected_points > 0
    GROUP BY position ORDER BY best DESC
""")

,position,players,best,median
0,QB,65,369.2,114.0
1,RB,111,365.3,76.6
2,WR,187,356.3,65.8
3,TE,98,242.1,38.6
4,P,32,191.5,160.5
5,K,32,171.6,141.7
6,DST,32,133.2,89.2


<a id="predictable"></a>
## 2. Is punting predictable at all?

Partly — but the cruel part is that what's *stable* isn't what *scores*.

Gross average is the most persistent thing a punter does (leg strength is real), and it barely
matters for fantasy. Volume is what scores, and it isn't really a property of the punter at all.
Games played — the single largest term in a season — is close to noise.

In [4]:
seasons = q("SELECT * FROM punter_seasons WHERE games >= 12")
nxt = seasons.copy()
nxt["season"] -= 1
pairs = seasons.merge(nxt, on=["player_id", "season"], suffixes=("", "_next"))

stability = pd.DataFrame({
    "stat": ["points", "points_per_game", "punts_per_game", "gross_average", "net_average",
             "in20_rate", "in10_rate", "touchback_rate", "fair_catch_rate", "returned_rate",
             "games"],
}).assign(
    year_over_year_r=lambda d: [round(pairs[s].corr(pairs[f"{s}_next"]), 3) for s in d.stat]
).sort_values("year_over_year_r", ascending=False)

print(f"{len(pairs)} consecutive full-time punter-season pairs")
stability.reset_index(drop=True)

248 consecutive full-time punter-season pairs


,stat,year_over_year_r
0,gross_average,0.487
1,punts_per_game,0.480
2,net_average,0.398
3,returned_rate,0.344
4,fair_catch_rate,0.289
5,points_per_game,0.272
6,points,0.262
7,in20_rate,0.214
8,in10_rate,0.167
9,games,0.088


Read that against what actually *correlates with season points* in the same season, and the tension
is obvious: gross average is the most predictable stat and one of the least valuable.

In [5]:
same_season = q("SELECT * FROM punter_seasons WHERE games >= 12")
pd.DataFrame({
    "component": ["punts", "games", "punts_per_game", "in10_rate", "in20_rate",
                  "fair_catch_rate", "gross_average", "touchback_rate"],
}).assign(
    r_with_season_points=lambda d: [
        round(same_season["points"].corr(same_season[s]), 3) for s in d.component]
).sort_values("r_with_season_points", ascending=False, key=abs).reset_index(drop=True)

,component,r_with_season_points
0,punts,0.698
1,punts_per_game,0.617
2,in10_rate,0.462
3,in20_rate,0.420
4,games,0.402
5,fair_catch_rate,0.203
6,touchback_rate,-0.086
7,gross_average,0.069


<a id="badteams"></a>
## 3. Are bad teams' punters better?

Yes — but the mechanism isn't "more punts". It's **where the drives die**, and the effect is about
tenfold.

A punt is only worth much if it can be dropped inside the 20. A punter pinned on his own goal line
can't do that: he has to hit it as far as he can and live with the return. Note the top row —
a punt from inside your own 10 is worth **less than the single point a punt pays**, because
two-thirds of them come back.

In [6]:
q("""
WITH punts AS (
    SELECT
        100 - yardline_100 AS own_yard, kick_distance,
        CASE WHEN punt_inside_twenty = 1
              AND yardline_100 - kick_distance + COALESCE(return_yards, 0) < 10
             THEN 1 ELSE 0 END AS in10,
        COALESCE(punt_inside_twenty, 0) AS in20,
        COALESCE(touchback, 0) AS touchback,
        COALESCE(punt_fair_catch, 0) AS fair_catch,
        COALESCE(punt_blocked, 0) AS blocked,
        CASE WHEN punt_fair_catch = 1 OR touchback = 1 OR punt_downed = 1
                  OR punt_out_of_bounds = 1 OR punt_blocked = 1 OR punt_in_endzone = 1
             THEN 0 ELSE 1 END AS returned
    FROM pbp_punts
    WHERE season_type = 'REG' AND yardline_100 IS NOT NULL AND kick_distance IS NOT NULL
)
SELECT
    CASE WHEN own_yard <= 10 THEN 'own 1-10'  WHEN own_yard <= 20 THEN 'own 11-20'
         WHEN own_yard <= 30 THEN 'own 21-30' WHEN own_yard <= 40 THEN 'own 31-40'
         WHEN own_yard <= 50 THEN 'own 41-50' ELSE 'opponent side' END AS punt_from,
    COUNT(*) AS punts,
    ROUND(AVG(kick_distance), 1) AS gross,
    ROUND(AVG(in20), 3) AS in20_rate,
    ROUND(AVG(returned), 3) AS returned_rate,
    ROUND(1.0 + 3*AVG(in10) + 2*AVG(in20) - AVG(touchback) - AVG(returned)
              - AVG(blocked) + 0.5*AVG(fair_catch), 2) AS points_per_punt
FROM punts GROUP BY 1 ORDER BY MIN(own_yard)
""")

,punt_from,punts,gross,in20_rate,returned_rate,points_per_punt
0,own 1-10,1063,49.2,0.004,0.681,0.39
1,own 11-20,3177,49.0,0.014,0.636,0.48
2,own 21-30,5972,48.9,0.109,0.597,0.74
3,own 31-40,5779,48.3,0.415,0.474,1.67
4,own 41-50,4529,43.6,0.689,0.222,2.84
5,opponent side,3493,36.2,0.785,0.061,3.73


<a id="tension"></a>
## 4. Does punting more lower your average?

Yes — and it doesn't matter. Teams that punt most punt from deeper and lose roughly a tenth of
their value per punt, but they punt so much more often that the volume wins comfortably.

In [7]:
q("""
WITH ranked AS (
    SELECT *, NTILE(5) OVER (ORDER BY punts_per_game) AS bucket FROM punt_environment
)
SELECT
    CASE bucket WHEN 1 THEN '1 fewest punts' WHEN 5 THEN '5 most punts'
                ELSE CAST(bucket AS VARCHAR) END AS quintile,
    ROUND(AVG(punts_per_game), 2)       AS punts_per_game,
    ROUND(AVG(avg_punt_spot), 1)        AS avg_punt_spot,
    ROUND(AVG(points_per_punt), 2)      AS points_per_punt,
    ROUND(AVG(punt_points_per_game), 2) AS punt_points_per_game,
    ROUND(AVG(points_for), 1)           AS team_points_per_game
FROM ranked GROUP BY bucket ORDER BY bucket
""")

,quintile,punts_per_game,avg_punt_spot,points_per_punt,punt_points_per_game,team_points_per_game
0,1 fewest punts,3.03,35.0,1.85,5.60,27.1
1,2,3.64,34.3,1.77,6.45,24.1
2,3,4.14,33.9,1.75,7.25,22.0
3,4,4.59,34.2,1.77,8.14,21.5
4,5 most punts,5.33,33.4,1.64,8.72,19.5


The sharpest way to see it is within a single volume tier. These team-seasons all punted at
roughly the same rate — what separates their punters is entirely where the punts came from.

In [8]:
q("""
    SELECT team, season,
           ROUND(punts_per_game, 2)       AS punts_per_game,
           ROUND(avg_punt_spot, 1)        AS avg_punt_spot,
           ROUND(points_per_punt, 2)      AS points_per_punt,
           ROUND(punt_points_per_game, 2) AS punt_points_per_game
    FROM punt_environment
    WHERE punts_per_game > 5.5
    ORDER BY punt_points_per_game DESC
""")

,team,season,punts_per_game,avg_punt_spot,points_per_punt,punt_points_per_game
0,LA,2016,6.13,31.8,2.15,13.19
1,SF,2016,6.25,32.6,1.69,10.53
2,NYJ,2023,5.82,33.6,1.73,10.06
3,NYJ,2017,5.88,33.0,1.68,9.84
4,NYG,2023,5.59,32.7,1.73,9.65
5,HOU,2017,5.75,31.1,1.65,9.50
6,MIA,2016,5.63,34.0,1.69,9.50
7,NE,2023,5.76,32.8,1.61,9.29
8,NYG,2016,5.81,34.8,1.59,9.25
9,ARI,2018,5.88,32.6,1.55,9.13


<a id="matchup"></a>
## 5. Does matchup matter?

It's real, and nearly as large as *which punter you own* — but it doesn't survive testing.

Pooled over every season, the gap between the most and least punter-friendly defense is a couple of
points a game. The problem is that it doesn't persist: a defense's punter-points-allowed carries
year to year at only about r=0.16, and out of sample, matchup-aware weekly rankings score no better
than simply knowing how often a punter's own team punts.

Weather is weaker still — dome and outdoor punters score within a fifth of a point of each other,
and wind buckets don't order at all. (Cold does cut gross average; it just doesn't cut points.)

In [9]:
matchup = q("""
    SELECT team AS defense,
           ROUND(SUM(punts_forced_per_game * games) / SUM(games), 2) AS punts_forced_per_game,
           ROUND(SUM(punt_points_allowed_per_game * games) / SUM(games), 2) AS points_allowed_per_game
    FROM punt_environment GROUP BY team ORDER BY points_allowed_per_game DESC
""")
print(f"spread across 32 defenses: "
      f"{matchup.points_allowed_per_game.max() - matchup.points_allowed_per_game.min():.2f} "
      f"points per game")
pd.concat([matchup.head(5), matchup.tail(5)])

spread across 32 defenses: 2.44 points per game


,defense,punts_forced_per_game,points_allowed_per_game
0,DEN,4.68,8.60
1,NYJ,4.50,8.54
2,LAC,3.93,8.20
3,HOU,4.50,7.88
4,JAX,4.24,7.80
27,CIN,4.06,6.49
28,DET,3.75,6.47
29,NO,4.02,6.45
30,KC,3.90,6.34
31,LV,3.68,6.16


<a id="model"></a>
## 6. The model, and how well it does

`src/gold/punters.py` builds the projection. It's empirical-Bayes shrinkage rather than a learner,
because there are only ~250 usable consecutive punter-season pairs and the rates barely persist —
a gradient booster would fit noise. Two structural choices do the real work:

- **Volume comes from the team, not the punter.** For punters who changed teams, their new team's
  prior punt rate predicts them better than their own history does.
- **Expected games comes from incumbency.** Whether the job is already his, which is knowable at
  draft time and worth ~33 points.

Walk-forward against three naive baselines:

In [10]:
q("""
    SELECT prediction,
           ROUND(AVG(mae), 1)      AS mae,
           ROUND(AVG(rmse), 1)     AS rmse,
           ROUND(AVG(spearman), 3) AS rank_r,
           CAST(SUM(top10_hits) AS INT) || '/' || CAST(COUNT(*) * 10 AS INT) AS top10_hits,
           COUNT(*) AS seasons
    FROM punter_backtest GROUP BY prediction ORDER BY mae
""")

,prediction,mae,rmse,rank_r,top10_hits,seasons
0,model,38.5,50.9,0.331,25/70,7
1,league_mean,40.3,56.3,NaN,None,7
2,prior_ppg,41.2,54.6,0.148,23/70,7
3,prior_points,42.0,56.1,0.259,26/70,7


The model leads on every metric that has an answer. But the honest read is the ceiling: handed the
true games count its MAE drops to ~20 and rank correlation to ~0.67. **Nearly all the remaining
error is not knowing who keeps the job in November** — and no method, including the naive ones,
reliably picks the top five.

Per-season, so you can see how much it swings:

In [11]:
q("""
    SELECT target_season,
           ROUND(MAX(CASE WHEN prediction = 'model'        THEN mae END), 1) AS model,
           ROUND(MAX(CASE WHEN prediction = 'prior_points' THEN mae END), 1) AS prior_points,
           ROUND(MAX(CASE WHEN prediction = 'prior_ppg'    THEN mae END), 1) AS prior_ppg,
           ROUND(MAX(CASE WHEN prediction = 'league_mean'  THEN mae END), 1) AS league_mean
    FROM punter_backtest GROUP BY target_season ORDER BY target_season
""")

,target_season,model,prior_points,prior_ppg,league_mean
0,2019,38.4,41.4,39.4,41.0
1,2020,37.2,38.7,36.5,43.8
2,2021,41.6,41.9,38.2,47.0
3,2022,43.6,48.1,45.6,34.2
4,2023,36.5,43.7,45.6,37.8
5,2024,42.3,47.4,47.2,45.4
6,2025,29.7,33.2,35.8,33.2


<a id="projections"></a>
## 7. This year's projections

`projected_points` is this model; `projected_points_espn` is ESPN's own, kept as its own column so
the two can disagree visibly. ESPN publishes nothing for punters it isn't projecting, which shows
up as null rather than zero.

`incumbency` is the column to look at hardest — it's the largest single swing on the board.

In [12]:
q("""
    SELECT player_name, team, incumbency,
           ROUND(projected_points, 1)         AS model,
           ROUND(projected_points_espn, 1)    AS espn,
           ROUND(expected_games, 1)           AS exp_games,
           ROUND(expected_punts, 1)           AS exp_punts,
           ROUND(gross_average, 1)            AS gross,
           ROUND(in20_rate, 3)                AS in20_rate
    FROM punter_projections
    ORDER BY projected_points DESC LIMIT 15
""")

,player_name,team,incumbency,model,espn,exp_games,exp_punts,gross,in20_rate
0,Austin McNamara,NYJ,incumbent,166.9,176.3,15.3,66.5,46.4,0.406
1,Sam Martin,CAR,incumbent,164.5,185.6,15.3,63.3,46.4,0.433
2,Logan Cooke,JAX,incumbent,159.4,169.5,15.3,61.4,47.5,0.420
3,Jack Fox,DET,incumbent,157.3,154.6,15.3,58.9,47.1,0.438
4,AJ Cole,LV,incumbent,156.2,181.1,15.3,63.7,48.3,0.390
5,Mitch Wishnowsky,BUF,incumbent,153.9,NaN,15.3,57.5,46.2,0.432
6,J.K. Scott,LAC,incumbent,153.6,162.9,15.3,63.5,46.7,0.385
7,Corey Bojorquez,CLE,incumbent,153.0,177.7,15.3,71.0,47.4,0.345
8,Ryan Rehkow,CIN,incumbent,152.6,154.4,15.3,62.4,48.2,0.414
9,Riley Dixon,TB,incumbent,152.0,154.9,15.3,61.3,46.0,0.399


In [13]:
q("""
    SELECT incumbency, COUNT(*) AS punters,
           ROUND(AVG(projected_points), 1) AS avg_projection
    FROM punter_projections GROUP BY incumbency ORDER BY avg_projection DESC
""")

,incumbency,punters,avg_projection
0,incumbent,19,150.9
1,unproven,6,129.6
2,new_team,10,114.1


---

## Adding to this notebook

`q()` opens a read-only connection, runs, and closes it — so nothing here can hold a lock that
blocks `scripts/build_warehouse.sh`, and nothing here can write to the warehouse. See
`notebooks/README.md` for the conventions, and `src/query.py` for why it works that way.

Useful while exploring:

```python
tables()              # every table with row counts
tables("gold")        # just the models
columns("punter_seasons")
peek("punt_environment")
```